# Personal Knowledge Assistant


## Step 1 — Install

In [ ]:
!pip install crewai ddgs sentence-transformers pypdf litellm -q
!pip install -U pillow -q

## Step 1b — Setup

In [ ]:
import asyncio, os
import numpy as np
import litellm, litellm.main
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from google.colab import userdata, files

# Patch: Groq "cache_breakpoint" accept nahi karta
if not getattr(litellm, "_patched_ok", False):
    _real = litellm.main.completion
    def _clean(messages):
        return [{k: v for k, v in m.items() if k != "cache_breakpoint"} if isinstance(m, dict) else m for m in messages]
    def _patched(*args, **kwargs):
        if kwargs.get("messages"):
            kwargs["messages"] = _clean(kwargs["messages"])
        return _real(*args, **kwargs)
    litellm.completion = _patched
    litellm._patched_ok = True

llm = LLM(model="groq/openai/gpt-oss-20b", api_key=userdata.get("GROQ_API_KEY"), is_litellm=True)
print("Setup theek hai")

Setup theek hai


## Step 2 — Documents upload (notes.txt chunein)

In [ ]:
if not os.path.exists("notes.txt"):
    files.upload()
with open("notes.txt", "r", encoding="utf-8") as f:
    document_text = f.read()
print(document_text[:300])

Git and GitHub Notes

Git is a version control tool that runs on your own computer. It saves checkpoints of your project called commits. Every commit has a short message that describes what changed.

GitHub is a website that stores a copy of your Git repository online. You upload commits to GitHub u


## Step 3 — Chunk aur Embed

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(document_text)
print(f"Created {len(chunks)} chunks")

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks)

Created 4 chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Step 4 — Retrieval test

In [ ]:
def retrieve(query, top_k=3):
    q = embedder.encode([query])[0]
    sims = chunk_embeddings @ q / (np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q))
    best = np.argsort(sims)[::-1][:top_k]
    return [chunks[i] for i in best]

for r in retrieve("What is a merge conflict?"):
    print(r, "\n---")

fe copy where you can experiment. The main branch is the official version of the project.

Merging means combining the work of one branch into another branch. When two people change the same line of the same file, Git reports a merge conflict and asks you to choose which version to keep.

The rhythm to remember is: pull, branch, work, commit, push.

Agentic AI Notes

An AI agent is a language model that has a role, a goal, and sometimes tools that it can call. A crew is a group of agents that wo 
---
t it can call. A crew is a group of agents that work together on tasks one after another.

RAG stands for Retrieval Augmented Generation. In RAG, a document is split into small chunks, each chunk is converted into an embedding (a list of numbers that captures meaning), and the chunks most similar to a question are retrieved and given to the language model.

A tool is a Python function that an agent can call, for example a web search function. Giving an agent a tool it never uses is a desig

### ✅ COMMIT 1 — `RAG pipeline working — documents chunked, embedded, and retrievable`

## Step 5 — Document Search tool aur Researcher

In [ ]:
@tool("Document Search")
def document_search(query: str) -> str:
    """Searches the student's own uploaded documents for relevant passages."""
    return str(retrieve(query))

researcher = Agent(
    role="Personal Notes Researcher",
    goal="Find the passages in the student's own uploaded documents that are most relevant to the question.",
    backstory="You only read the student's own notes, never the open web, and never invent facts.",
    tools=[document_search], llm=llm)

## Step 6 — Web Search tool aur Verifier

In [ ]:
@tool("Web Search")
def web_search(query: str) -> str:
    """Searches the web for current information on a given query."""
    from ddgs import DDGS
    return str(DDGS().text(query, max_results=3))

verifier = Agent(
    role="Fact Verifier",
    goal="Check the claims found in the student's notes against current information on the live web.",
    backstory="You are a skeptical fact-checker. You always search the web and say whether each claim is confirmed, outdated, or wrong.",
    tools=[web_search], llm=llm)

### ✅ COMMIT 2 — `Researcher and Verifier agents working against your documents and the web`

## Step 7 — Writer (koi tool nahi) aur Crew

In [ ]:
writer = Agent(
    role="Answer Writer",
    goal="Write one clear, accurate final answer using the retrieved notes and the verification result.",
    backstory="You turn research and fact-check results into a short, easy answer. If the web contradicts the notes, you say so plainly.",
    llm=llm)

research_task = Task(
    description="The student asked: '{question}'. Use the Document Search tool to find the passages in the student's own documents that answer this. List the relevant passages and key claims.",
    agent=researcher,
    expected_output="Relevant passages from the notes and the key factual claims in them.")

verify_task = Task(
    description="Take the key claims found by the Researcher for the question '{question}'. You MUST call the Web Search tool for at least one claim and compare with the notes. For each claim say CONFIRMED, OUTDATED, or WRONG with a short reason.",
    agent=verifier, context=[research_task],
    expected_output="Each claim marked CONFIRMED / OUTDATED / WRONG with a short web-based reason.")

write_task = Task(
    description="Write the final answer to the question '{question}' using the Researcher's notes and the Verifier's result. If the web contradicts the notes, mention the correction clearly.",
    agent=writer, context=[verify_task],
    expected_output="One clear final answer in 1-2 short paragraphs.")

crew = Crew(
    agents=[researcher, verifier, writer],
    tasks=[research_task, verify_task, write_task],
    process=Process.sequential,
    max_rpm=5)

# Groq ki limit aaye to khud intezar karke dobara chalata hai
async def ask(q):
    for attempt in range(5):
        try:
            return await crew.kickoff_async(inputs={"question": q})
        except Exception as e:
            if "rate" in str(e).lower():
                print(f"Rate limit aya (koshish {attempt + 1}). 65 second intezar...")
                await asyncio.sleep(65)
            else:
                raise

result = await ask("What is a merge conflict and how do I resolve it?")
print("\nFINAL ANSWER:\n")
print(result.raw)


FINAL ANSWER:

A merge conflict happens when Git cannot automatically combine changes from two branches because the same lines in a file were edited differently in each branch. Git’s merge algorithm detects this situation, marks the file with conflict markers (`<<<<<<<`, `=======`, `>>>>>>>`), and pauses the merge so you can decide how to proceed.

To resolve the conflict, open the conflicted file, edit the marked sections to keep the desired changes, and then remove the conflict markers. After saving the file, mark the conflict as resolved (e.g., `git add <file>`), and continue the merge or commit the resolution.


### ✅ COMMIT 3 — `Full 3-agent crew producing answers end to end`

## Step 8 — Memory on karein

In [ ]:
crew = Crew(
    agents=[researcher, verifier, writer],
    tasks=[research_task, verify_task, write_task],
    process=Process.sequential,
    max_rpm=5,
    memory=True,
    embedder={
        "provider": "sentence-transformer",
        "config": {"model_name": "all-MiniLM-L6-v2"}
    })
print("Crew with memory ban gaya")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Crew with memory ban gaya


### Run 1 (pehla sawal)

In [ ]:
r1 = await ask("What is RAG and how does it work?")
print(r1.raw)

ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}




Retrieval‑augmented generation (RAG) is a method that lets a large language model answer questions by pulling in fresh, external information. In a RAG pipeline, documents are first split into small chunks, each chunk is turned into an embedding that captures its meaning, and the collection of embeddings is indexed. When a user asks a question, the system performs a vector search over those embeddings, retrieves the chunks most similar to the query, and feeds those chunks into the language model. The model then generates an answer that is conditioned on the retrieved content, giving it up‑to‑date, context‑specific knowledge.


### Run 2 (pehle se mutalliq sawal) — Run 1 ke 1 minute baad chalayein

In [ ]:
r2 = await ask("Why do we split documents into chunks in that process?")
print(r2.raw)

ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}


ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}


ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"in

In Retrieval‑Augmented Generation (RAG) we break a long document into smaller, self‑contained chunks before encoding. Each chunk is turned into its own embedding, which lets the system quickly compare the query against many compact vectors rather than one huge one. This speeds retrieval, keeps the computational load manageable, and allows the model to focus only on the most relevant sections when generating an answer.


### Run 3 (purani/galat baat ka test — Python version)

In [ ]:
r3 = await ask("What is the latest stable version of Python?")
print(r3.raw)

ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}


ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}


ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"in

The latest stable release of Python is **Python 3.14.6**, which was published in early October 2026.  A newer point release, **Python 3.14.7**, is slated for release shortly thereafter and will become the new stable version.


### ✅ COMMIT 4 — `Memory enabled — two related runs show the crew recalling context`

Phir README likhein → **COMMIT 5** — `README and sample Q&A added — final submission`